# Домашнее задание №6
## Среднеквадратическая аппроксимация табличной функции методом наименьших квадратов

ису: 501290

я взяла ненулевые цифры ИСУ 501290 по порядку: $5, 1, 2, 9$. их четыре, для полинома степени $4$ нужно пять коэффициентов, поэтому свободный член задала равным $1$. получила полином с ненулевыми коэффициентами:

$$f(x) = 5x^{4} + x^{3} + 2x^{2} + 9x + 1.$$

---

## 1. задача

по условию нужно:

1. для полинома степени $m = 4$ с ненулевыми коэффициентами сгенерировать датасет из $N$ точек;
2. лля $N \gg m$ применить метод наименьших квадратов (среднеквадратическую аппроксимацию), построить получившийся полином и оценить точность приближения;
3. добавить весовые коэффициенты, определяемые методом экспертной оценки.

функция $f(x)$ задана на дискретном множестве точек $x_1, x_2, \dots, x_N$. её аппроксимация ищется в виде обобщённого многочлена:

$$Q_m(x) = a_0 \varphi_0(x) + a_1 \varphi_1(x) + \dots + a_m \varphi_m(x),$$

где $\{\varphi_k\}$ это набор линейно независимых базисных функций 
для аппроксимации полиномами $\varphi_k(x) = x^{k}$.

коэффициенты $a_k$ выбираются из условия минимума величины $\rho^2$:

$$\rho^2 = \sum_{i=1}^{N} \big(Q(x_i) - f(x_i)\big)^2 \to \min.$$

записывая необходимое условие экстремума $\partial \rho^2 / \partial a_k = 0$, приходим к системе линейных алгебраических уравнений относительно $a_k$:

$$\sum_{j=0}^{m} \Big(\sum_{i=1}^{N} x_i^{k} x_i^{j}\Big) a_j = \sum_{i=1}^{N} x_i^{k} f(x_i), \qquad k = 0, 1, \dots, m.$$

в матричном виде это нормальная система $A a = b$, где элементы $A_{kj} = \sum_i x_i^{k} x_i^{j} = (\varphi_k, \varphi_j)$, а $b_k = \sum_i x_i^{k} f(x_i)$. по усл $N > m + 1$ задача имеет единственное решение.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. генерация датасета с погрешностью

полином:

$$f(x) = 5x^{4} + x^{3} + 2x^{2} + 9x + 1.$$

точки $x$ берём равномерно на отрезке $[-1, 1]$, как в примере из условия. Число точек $N = 50$, что много больше степени $m = 4$ (то есть выполнено $N \gg m$).

погрешность измерений моделируеься равномерным шумом небольшой амплитуды. равномерный шум выбран потому, что условие рекомендует избегать методов, создающих экстремальные выбросы, а равномерное распределение не порождает редких больших отклонений.

In [ ]:
np.random.seed(501290)

def f(x):
    return 5*x**4 + 1*x**3 + 2*x**2 + 9*x + 1

m = 4
N = 50 

x_data = np.linspace(-1, 1, N)

y_true = f(x_data)

noise = np.random.uniform(-0.5, 0.5, N)

y_data = y_true + noise

print('N =', N, ', m =', m)
print('размах функции на [-1,1]:', round(y_true.max() - y_true.min(), 2))
print()
print('первые 5 точек датасета:')
for i in range(5):
    print(f'  x = {x_data[i]:6.3f},  y_изм = {y_data[i]:7.3f}')

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(x_data, y_true, 'g-', label='истинный полином f(x)')
plt.scatter(x_data, y_data, s=20, color='red', label='измерения с погрешностью')
plt.xlabel('x')
plt.ylabel('y')
plt.title('сгенерированный датасет')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. метод наименьших квадратов

базис полиномиальной аппроксимации: $\varphi_k(x) = x^{k}$, $k = 0, 1, \dots, m$. составим матрицу $\Phi$ размера $N \times (m+1)$, в которой элемент $\Phi_{ik} = x_i^{k}$

In [ ]:
Phi = np.zeros((N, m+1))
for k in range(m+1):
    Phi[:, k] = x_data**k

print('размер матрицы Phi:', Phi.shape)
print('первые 3 строки (столбцы = x^0, x^1, x^2, x^3, x^4):')
print(np.round(Phi[:3], 3))

нормальная система имеет вид $A a = b$, где

$$A = \Phi^{T} \Phi, \qquad b = \Phi^{T} y.$$

элемент $A_{kj} = \sum_i x_i^{k} x_i^{j}$ это скалярное произведение $(\varphi_k, \varphi_j)$, а $b_k = \sum_i x_i^{k} y_i$.

In [ ]:
A = Phi.T @ Phi
b = Phi.T @ y_data

print('матрица нормальной системы A = Phi^T Phi (размер {}x{}):'.format(*A.shape))
print(np.round(A, 3))
print()
print('правая часть b = Phi^T y:')
print(np.round(b, 3))


СЛАУ $A a = b$ и коэффициенты аппроксимирующего полинома $a_0, a_1, \dots, a_m$.

In [ ]:
a = np.linalg.solve(A, b)

print('найденные коэффициенты аппроксимирующего полинома:')
for k in range(m+1):
    print(f'  a_{k} = {a[k]:.4f}')
print()
print('истинные коэффициенты полинома по ИСУ:')
print('  a_0 = 1, a_1 = 9, a_2 = 2, a_3 = 1, a_4 = 5')

### построение полинома

значения аппроксимирующего полинома: $Q(x) = \Phi a$. 

In [ ]:
y_approx = Phi @ a

plt.figure(figsize=(9, 5))
plt.scatter(x_data, y_data, s=20, color='red', label='измерения с погрешностью')
plt.plot(x_data, y_true, 'g-', label='истинный полином f(x)')
plt.plot(x_data, y_approx, 'b--', label='аппроксимация Q(x) по МНК')
plt.xlabel('x')
plt.ylabel('y')
plt.title('среднеквадратическая аппроксимация')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### оценка точности приближения

в качестве меры точности использую величину $\rho^2$ (сумму квадратов отклонений аппроксимации от измеренных значений) и среднеквадратичное отклонение $\sqrt{\rho^2 / N}$.

In [ ]:
rho2 = np.sum((y_approx - y_data)**2)

rms = np.sqrt(rho2 / N)

print(f'rho^2 = sum (Q(xi) - y_i)^2 = {rho2:.4f}')
print(f'среднеквадратичное отклонение sqrt(rho^2 / N) = {rms:.4f}')
print()
rho2_true = np.sum((y_approx - y_true)**2)
print(f'отклонение Q(x) от истинного f(x): sum (Q(xi)-f(xi))^2 = {rho2_true:.4f}')

## 4. весовые коэффициенты, метод экспертной оценки


$$\rho^2 = \sum_{i=1}^{N} p_i \big(Q(x_i) - f(x_i)\big)^2 \to \min.$$
нормальная система с весами принимает вид:

$$\Phi^{T} P \, \Phi \, a = \Phi^{T} P \, y,$$

где $P = \mathrm{diag}(p_1, \dots, p_N)$ это диагональная матрица весов.


In [ ]:
weights = 1.0 + np.exp(-x_data**2 * 5)

weights = weights / weights.sum() * 100

print('веса в отдельных точках:')
print(f'  левый край:  p[0]  = {weights[0]:.3f}')
print(f'  центр:       p[25] = {weights[25]:.3f}')
print(f'  правый край: p[-1] = {weights[-1]:.3f}')
print(f'  сумма весов = {weights.sum():.2f}')

### 5.3. Решение взвешенной нормальной системы

In [ ]:
P = np.diag(weights)

A_w = Phi.T @ P @ Phi
b_w = Phi.T @ P @ y_data

a_w = np.linalg.solve(A_w, b_w)

print('коэффициенты с учётом весов:')
for k in range(m+1):
    print(f'  a_{k} = {a_w[k]:.4f}')

### 5.4. Сравнение приближений без весов и с весами

In [ ]:
print(f"{'k':<4}{'без весов':<14}{'с весами':<14}{'истина':<10}")
true_coeffs = [1, 9, 2, 1, 5]
for k in range(m+1):
    print(f'{k:<4}{a[k]:<14.4f}{a_w[k]:<14.4f}{true_coeffs[k]:<10}')

In [ ]:
y_approx_w = Phi @ a_w

plt.figure(figsize=(9, 5))
plt.scatter(x_data, y_data, s=20, color='red', label='измерения с погрешностью')
plt.plot(x_data, y_approx,   'b--', label='МНК без весов')
plt.plot(x_data, y_approx_w, 'm-',  label='МНК с весами')
plt.xlabel('x')
plt.ylabel('y')
plt.title('сравнение аппроксимации без весов и с весами')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()